# Clase 2 · Laboratorio — Diseño dimensional para Saber 11

**Trabajo en parejas · 90 min.**

Al final deben tener un notebook completado con las 4 tareas resueltas y subirlo a Moodle antes de las 23:59.

**Checkpoint conjunto a los 55 min:** el profesor detiene la sala y revisamos juntos la Tarea 3 (hecho + FKs) antes de pasar al diseño del proyecto propio.

In [1]:
import pandas as pd

df = pd.read_csv("../datos/saber11_muestra_500k.csv")
print(f"Cargados {len(df):,} registros — {df.shape[1]} columnas")

Cargados 500,000 registros — 51 columnas


## Tarea 1 (15 min) — dim_colegio con clave surrogate

Construye una tabla de dimensión `dim_colegio` que contenga:
- Una clave surrogate `colegio_id` (entero secuencial).
- Los atributos: COLE_NATURALEZA, COLE_JORNADA, COLE_CALENDARIO, COLE_BILINGUE.

Requisitos:
- Sin duplicados (una fila por combinación única de los 4 atributos).
- Verifica que la clave surrogate es única.

In [2]:
# TODO: construir dim_colegio con las 4 columnas indicadas y una clave surrogate `colegio_id`.
cols_colegio = ["COLE_NATURALEZA", "COLE_JORNADA", "COLE_CALENDARIO", "COLE_BILINGUE"]

dim_colegio = df[cols_colegio].drop_duplicates().reset_index(drop=True)

dim_colegio.insert(0, "colegio_id", dim_colegio.index + 1)


# Validar unicidad de la surrogate
assert dim_colegio["colegio_id"].is_unique, "La clave surrogate NO es única"
print(f"dim_colegio: {len(dim_colegio)} filas")
dim_colegio.head()

dim_colegio: 56 filas


,colegio_id,COLE_NATURALEZA,COLE_JORNADA,COLE_CALENDARIO,COLE_BILINGUE
0,1,OFICIAL,COMPLETA,A,N
1,2,OFICIAL,MAÑANA,A,N
2,3,OFICIAL,SABATINA,A,NaN
3,4,NO OFICIAL,SABATINA,A,NaN
4,5,OFICIAL,COMPLETA,A,NaN


## Tarea 2 (15 min) — dim_geografia con jerarquía

Construye `dim_geografia` con:
- Clave surrogate `geo_id`.
- Atributos: COLE_DEPTO_UBICACION (padre), COLE_MCPIO_UBICACION (hijo).

Requisitos:
- Una fila por par (departamento, municipio).
- Documenta en Markdown por qué la jerarquía es útil para el análisis.

In [3]:
# TODO: dim_geografia
cols_geo = ["COLE_DEPTO_UBICACION", "COLE_MCPIO_UBICACION"]

dim_geografia = df[cols_geo].drop_duplicates().reset_index(drop=True)

dim_geografia.insert(0, "geo_id", dim_geografia.index + 1)

assert dim_geografia["geo_id"].is_unique
print(f"dim_geografia: {len(dim_geografia)} filas")
dim_geografia.head()

dim_geografia: 1114 filas


,geo_id,COLE_DEPTO_UBICACION,COLE_MCPIO_UBICACION
0,1,HUILA,AIPE
1,2,HUILA,LA PLATA
2,3,SANTANDER,BUCARAMANGA
3,4,ANTIOQUIA,ITAGÜÍ
4,5,CAQUETA,LA MONTAÑITA


## Tarea 3 (25 min) — hecho_resultados

Construye la tabla de hechos `hecho_resultados` con:
- FKs a: `dim_colegio` (colegio_id), `dim_geografia` (geo_id), `dim_tiempo` (tiempo_id — bosqueja también esta dimensión).
- Medidas: PUNT_LECTURA_CRITICA, PUNT_MATEMATICAS, PUNT_C_NATURALES, PUNT_SOCIALES_CIUDADANAS, PUNT_INGLES, PUNT_GLOBAL.

Requisitos:
- La granularidad del hecho es "una fila por estudiante-periodo".
- Después de hacer los joins con las dimensiones, la tabla de hechos NO debe perder filas (comparar `len(hecho_resultados)` con `len(df)` original).

In [4]:
# 1) dim_tiempo (boceto)
dim_tiempo = df[["PERIODO"]].drop_duplicates().reset_index(drop=True)
dim_tiempo["tiempo_id"] = dim_tiempo.index + 1

# 2) TODO: unir df con las 3 dimensiones para obtener las FKs
hecho = df.merge(
    dim_colegio, 
    on=["COLE_NATURALEZA", "COLE_JORNADA", "COLE_CALENDARIO", "COLE_BILINGUE"], 
    how="left"
)
hecho = hecho.merge(
    dim_geografia, 
    on=["COLE_DEPTO_UBICACION", "COLE_MCPIO_UBICACION"], 
    how="left"
)
hecho = hecho.merge(
    dim_tiempo, 
    on="PERIODO", 
    how="left"
)
# Seleccionar FKs y medidas solicitadas
fks = ["colegio_id", "geo_id", "tiempo_id"]
medidas = [
    "PUNT_LECTURA_CRITICA", 
    "PUNT_MATEMATICAS", 
    "PUNT_C_NATURALES", 
    "PUNT_SOCIALES_CIUDADANAS", 
    "PUNT_INGLES", 
    "PUNT_GLOBAL"
]
hecho_resultados = hecho[fks + medidas].copy()

# 3) validar que no perdimos filas
assert len(hecho_resultados) == len(df), f"Perdimos filas: {len(df) - len(hecho_resultados)}"
print(f"hecho_resultados: {len(hecho_resultados)} filas (esperado {len(df)})")
hecho_resultados.head()

hecho_resultados: 500000 filas (esperado 500000)


,colegio_id,geo_id,tiempo_id,PUNT_LECTURA_CRITICA,PUNT_MATEMATICAS,PUNT_C_NATURALES,PUNT_SOCIALES_CIUDADANAS,PUNT_INGLES,PUNT_GLOBAL
0,1,1,1,69,66,65,70,71.0,339
1,1,1,1,69,66,65,70,71.0,339
2,1,2,1,43,43,40,31,46.0,199
3,1,2,1,43,43,40,31,46.0,199
4,2,3,1,63,64,59,49,61.0,295


## ⏸ Checkpoint del profesor (10 min)

Detengan aquí. El profesor revisa la Tarea 3 con toda la sala: cómo evitar perder filas al hacer joins, qué hacer si aparecen nulos en las FKs.

## Tarea 4 (30 min) — Modelo dimensional del proyecto del grupo

Con tu grupo, diseñen el modelo dimensional para su dataset del proyecto:

1. Identifiquen el **hecho** principal y su granularidad.
2. Identifiquen 3-4 **dimensiones** y justifiquen brevemente cada una.
3. Dibujen el modelo (celda Markdown con diagrama tipo Mermaid, ASCII, o adjunten un PNG de draw.io).

Al final, guarden el diagrama en el repo del grupo bajo `entrega/fase_a_diseño_borrador.pdf`.

### Diagrama de nuestro grupo

https://github.com/alejoneru/Ciencia_De_Datos_/
